In [1]:
!rm -rf diploma_centpy_parallelization_py
# Флаг -b указывает конкретную ветку
!git clone -b feature/jax-centpy https://github.com/filkinc/diploma_centpy_parallelization_py.git
%cd diploma_centpy_parallelization_py
%cd /content/diploma_centpy_parallelization_py/jax_centpy

# Установка зависимостей
!pip install centpy pandas matplotlib seaborn

Cloning into 'diploma_centpy_parallelization_py'...
remote: Enumerating objects: 247, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 247 (delta 2), reused 5 (delta 1), pack-reused 215 (from 1)
Receiving objects: 100% (247/247), 21.58 MiB | 7.29 MiB/s, done.
Resolving deltas: 100% (96/96), done.
/content/diploma_centpy_parallelization_py
/content/diploma_centpy_parallelization_py/jax_centpy


In [2]:
import os
import time
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import numpy as np
from google.colab import files

from core import Pars2d, Equation2d
from solver import Solver2d, FastSolver2d
from boundaries import periodic_bc_2d, neumann_bc_2d, dirichlet_riemann_bc_2d
from equations import make_euler_riemann_2d, make_euler_isentropic_vortex_2d
from schemes import compute_rhs_sd2_2d
from limiters import monotonized_central, minmod
from richardson import self_convergence_analysis

In [ ]:
# Основная функция проверки скорости работы схемы

def run_gpu_benchmark():
    print(f"JAX Device(s): {jax.devices()}")
    J = 200

    pars = Pars2d(
        x_init=0.0, x_final=1.0, y_init=0.0, y_final=1.0,
        t_final=0.4, dt_out=0.005, Jx=J, Jy=J, cfl=0.475, scheme="sd2"
    )

    eqn = make_euler_riemann_2d()
    solver = FastSolver2d(pars, eqn, limiter_name="minmod")
    solverForPlot = Solver2d(pars, eqn, scheme_name="sd2", limiter_name="minmod")

    print("--- Прогрев JAX (Компиляция XLA) ---")
    # Прогреваем ТОЛЬКО первый шаг решателя (именно он содержит всю тяжелую математику)
    x_1d = jnp.linspace(pars.x_init + pars.dx / 2, pars.x_final - pars.dx / 2, pars.Jx)
    y_1d = jnp.linspace(pars.y_init + pars.dy / 2, pars.y_final - pars.dy / 2, pars.Jy)
    X, Y = jnp.meshgrid(x_1d, y_1d, indexing='ij')
    test_u = eqn.initial_data(X, Y)
    _ = solver.update_step_jit(0.0, test_u, 0.001).block_until_ready()
    print("Прогрев завершен.\n")

    print(f"--- Запуск JAX GPU Бенчмарка (Euler 2D Riemann, {J}x{J}) ---")
    t0 = time.time()
    results = solver.solve()
    results['u'][-1].block_until_ready()
    t1 = time.time()

    print(f"\n[GPU JAX] Чистое время выполнения: {t1 - t0:.4f} секунд")

    resultsForPlot = solverForPlot.solve()

    return resultsForPlot, pars

if __name__ == "__main__":
    jax.config.update("jax_enable_x64", True)
    soln, pars = run_gpu_benchmark()

JAX Device(s): [CudaDevice(id=0)]
--- Прогрев JAX (Компиляция XLA) ---
Прогрев завершен.

--- Запуск JAX GPU Бенчмарка (Euler 2D Riemann, 200x200) ---
Starting 2D simulation: Euler 2D Riemann
Grid: 200x200, Scheme: SD2/minmod

[GPU JAX] Чистое время выполнения: 1.5267 секунд
Starting 2D simulation: Euler 2D Riemann
Grid: 200x200, Scheme: SD2/minmod


In [ ]:
# Приводим к единому стандарту: X, Y, t, u
# Берем данные из вашего словаря soln
np.savez('gpu_data.npz',
         X=np.array(soln['X']),
         Y=np.array(soln['Y']),
         u=np.array(soln['u'])) # Выгружаем весь массив u

# Скачиваем
files.download('gpu_data.npz')

In [ ]:
u_data = soln['u']
X_grid = soln['X']
Y_grid = soln['Y']

# Инициализация фигуры и осей, берем границы из объекта pars
fig, ax = plt.subplots()
ax.set_xlim(pars.x_init, pars.x_final)
ax.set_ylim(pars.y_init, pars.y_final)

# Для первого кадра (плотность, так как индекс [..., 0] соответствует плотности в уравнениях Эйлера)
data_init = u_data[0, ..., 0]

# 1. Создаем фоновую тепловую карту с помощью imshow
im = ax.imshow(
    data_init.T, # Транспонируем, так как imshow ожидает порядок (y, x), а у вас индексация 'ij'
    extent=[pars.x_init, pars.x_final, pars.y_init, pars.y_final],
    origin='lower',
    cmap='coolwarm',            # Золотой стандарт для волновых процессов; Или 'magma', 'viridis', 'turbo'
    interpolation='bicubic',
    aspect='auto'
)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Плотность (u[..., 0])')

# 2. Отрисовываем начальные контуры поверх тепловой карты
ax.contour(
    X_grid, Y_grid, data_init,
    levels=20,
    colors='black',
    alpha=0.5,
    linewidths=0.5
)

# Функция для сохранения картинки
def save_frame(frame_idx, filename):
    """Сохраняет указанный кадр как изображение"""
    fig_temp, ax_temp = plt.subplots()
    ax_temp.set_xlim(pars.x_init, pars.x_final)
    ax_temp.set_ylim(pars.y_init, pars.y_final)

    data = u_data[frame_idx, ..., 0]

    # Тепловая карта
    im_temp = ax_temp.imshow(
        data.T,
        extent=[pars.x_init, pars.x_final, pars.y_init, pars.y_final],
        origin='lower',
        cmap='coolwarm',
        interpolation='bicubic',
        aspect='auto'
    )

    cbar_temp = fig_temp.colorbar(im_temp, ax=ax_temp)
    cbar_temp.set_label('Плотность (u[..., 0])')

    # Контуры
    ax_temp.contour(
        X_grid, Y_grid, data,
        levels=20,
        colors='black',
        alpha=0.5,
        linewidths=0.5
    )

    # Сохраняем физический файл на диск перед скачиванием
    fig_temp.savefig(filename, bbox_inches='tight', dpi=150)

    # Закрываем фигуру, чтобы она не выводилась на экран в ячейке
    plt.close(fig_temp)

    files.download(filename);

# Сохраняем первый кадр
save_frame(0, 'first_frame.png');
print("Сохранен первый кадр: first_frame.png")

# Сохраняем последний кадр
last_frame_idx = u_data.shape[0] - 1
save_frame(last_frame_idx, 'last_frame.png');
print(f"Сохранен последний кадр (индекс {last_frame_idx}): last_frame.png")

# Функция обновления для каждого кадра анимации
def animate(i):
    # Получаем данные текущего шага
    data = u_data[i, ..., 0]

    # Обновляем данные на тепловой карте
    im.set_data(data.T)

    # Динамически обновляем границы цветовой шкалы (опционально)
    im.set_clim(vmin=data.min(), vmax=data.max())

    # Удаляем старые линии контуров из коллекции осей
    for c in ax.collections:
        c.remove()

    # Рисуем новые контурные линии
    ax.contour(
        X_grid, Y_grid, data,
        levels=20,
        colors='black',
        alpha=0.5,
        linewidths=0.5
    )

    return [im]

plt.close() # Закрываем статичную фигуру

# Создаем анимацию (количество кадров равно размеру массива времени)
num_frames = u_data.shape[0]
anim = animation.FuncAnimation(fig, animate, frames=num_frames, interval=100, blit=False)

video_filename = 'animation_euler.mp4'
anim.save(video_filename, writer='ffmpeg', fps=10) # fps можно настроить под нужную скорость
files.download(video_filename)
print(f"Сохранено и скачивается видео: {video_filename}")

# Закрываем основную фигуру, чтобы остался только видеоплеер
plt.close(fig)

# Выводим как HTML5 видео
HTML(anim.to_html5_video())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Сохранен первый кадр: first_frame.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Сохранен последний кадр (индекс 80): last_frame.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Сохранено и скачивается видео: animation_euler.mp4


In [ ]:
# Уравнение для проверки 2 порядка точности

def make_euler_isentropic_vortex_2d_colab(gamma: float = 1.4):
    """
    Гладкое точное решение: Изоэнтропийный вихрь.
    Идеально для проверки сходимости 2D Эйлера. Периодические граничные условия.
    Домен: x, y в [0, 10]. Центр вихря (5, 5). Скорость потока (1, 1).
    """

    def _compute_pressure(q):
        rho, u, v, E = q[..., 0], q[..., 1] / q[..., 0], q[..., 2] / q[..., 0], q[..., 3]
        return (gamma - 1.0) * (E - 0.5 * rho * (u ** 2 + v ** 2))

    def flux_x(q):
        rho, rhou, rhov, E = q[..., 0], q[..., 1], q[..., 2], q[..., 3]
        u = rhou / rho;
        p = _compute_pressure(q)
        return jnp.stack([rhou, rhou * u + p, rhou * (rhov / rho), u * (E + p)], axis=-1)

    def flux_y(q):
        rho, rhou, rhov, E = q[..., 0], q[..., 1], q[..., 2], q[..., 3]
        v = rhov / rho;
        p = _compute_pressure(q)
        return jnp.stack([rhov, rhov * (rhou / rho), rhov * v + p, v * (E + p)], axis=-1)

    def spectral_radius_x(q):
        rho, u, p = q[..., 0], q[..., 1] / q[..., 0], _compute_pressure(q)
        return jnp.abs(u) + jnp.sqrt(gamma * p / rho)

    def spectral_radius_y(q):
        rho, v, p = q[..., 0], q[..., 2] / q[..., 0], _compute_pressure(q)
        return jnp.abs(v) + jnp.sqrt(gamma * p / rho)

    # Функция генерирует точное решение для любого момента t
    def exact_solution(x, y, t=0.0):
        beta = 5.0
        # Смещение вихря с учетом скорости потока u=1, v=1 и периодичности
        dx = (x - 5.0 - t) % 10.0
        dy = (y - 5.0 - t) % 10.0
        dx = jnp.where(dx > 5.0, dx - 10.0, dx)
        dy = jnp.where(dy > 5.0, dy - 10.0, dy)

        r2 = dx ** 2 + dy ** 2
        du = - (beta / (2 * jnp.pi)) * jnp.exp(0.5 * (1.0 - r2)) * dy
        dv = (beta / (2 * jnp.pi)) * jnp.exp(0.5 * (1.0 - r2)) * dx

        u, v = 1.0 + du, 1.0 + dv
        T = 1.0 - ((gamma - 1.0) * beta ** 2 / (8 * gamma * jnp.pi ** 2)) * jnp.exp(1.0 - r2)
        rho = T ** (1.0 / (gamma - 1.0))
        p = rho ** gamma
        E = p / (gamma - 1.0) + 0.5 * rho * (u ** 2 + v ** 2)
        return jnp.stack([rho, rho * u, rho * v, E], axis=-1)

    return Equation2d(
        flux_x=flux_x, flux_y=flux_y,
        spectral_radius_x=spectral_radius_x, spectral_radius_y=spectral_radius_y,
        initial_data=lambda x, y: exact_solution(x, y, 0.0),
        boundary_handler=periodic_bc_2d,
        name="Euler 2D (Isentropic Vortex)"
    )

In [ ]:
# Проверки порядка аппроксимация экстраполяцией Ричардсона
# Использовал уравнение без ударной волны чтобы проверить что порядок будет 2
# Для уравнений с ударной волной порядок 2 не получить из за прямого следствия теоремы Годунова

def run_richardson_extrapolation(J1: int, J2: int, J3: int, equation: Equation2d, limiter: str):

  # Параметры для make_euler_isentropic_vortex_2d_colab() и limiter: "mc"
  # pars1 = Pars2d(
  #       x_init=0.0, x_final=10.0, y_init=0.0, y_final=10.0,
  #       t_final=5.0, dt_out=0.005, Jx=J1, Jy=J1, cfl=0.475, scheme="sd2"
  #   )

  # pars2 = Pars2d(
  #         x_init=0.0, x_final=10.0, y_init=0.0, y_final=10.0,
  #         t_final=5.0, dt_out=0.005, Jx=J2, Jy=J2, cfl=0.475, scheme="sd2"
  #     )

  # pars3 = Pars2d(
  #         x_init=0.0, x_final=10.0, y_init=0.0, y_final=10.0,
  #         t_final=5.0, dt_out=0.005, Jx=J3, Jy=J3, cfl=0.475, scheme="sd2"
  #     )

  # Параметры для make_euler_riemann_2d() и limiter: "mc"
  pars1 = Pars2d(
        x_init=0.0, x_final=1.0, y_init=0.0, y_final=1.0,
        t_final=0.4, dt_out=0.005, Jx=J1, Jy=J1, cfl=0.475, scheme="sd2"
    )

  pars2 = Pars2d(
        x_init=0.0, x_final=1.0, y_init=0.0, y_final=1.0,
        t_final=0.4, dt_out=0.005, Jx=J2, Jy=J2, cfl=0.475, scheme="sd2"
    )

  pars3 = Pars2d(
        x_init=0.0, x_final=1.0, y_init=0.0, y_final=1.0,
        t_final=0.4, dt_out=0.005, Jx=J3, Jy=J3, cfl=0.475, scheme="sd2"
    )

  eqn = equation
  solver1 = FastSolver2d(pars1, eqn, limiter_name=limiter)
  u_c = solver1.solve()

  solver2 = FastSolver2d(pars2, eqn, limiter_name=limiter)
  u_m = solver2.solve()

  solver3 = FastSolver2d(pars3, eqn, limiter_name=limiter)
  u_f = solver3.solve()


  euler_vars = ["Density (rho)", "Momentum X (rho*u)", "Momentum Y (rho*v)", "Energy (E)"]

  results = self_convergence_analysis(
      u_coarse=u_c["u"][-1],
      u_medium=u_m["u"][-1],
      u_fine=u_f["u"][-1],
      r=2.0,
      var_names=euler_vars
  )

if __name__ == "__main__":
    jax.config.update("jax_enable_x64", True)
    #run_richardson_extrapolation(50, 100, 200, make_euler_isentropic_vortex_2d_colab(), "mc")
    run_richardson_extrapolation(50, 100, 200, make_euler_riemann_2d(), "mc")


Starting 2D simulation: Euler 2D Riemann
Grid: 50x50, Scheme: SD2/monotonized_central
Starting 2D simulation: Euler 2D Riemann
Grid: 100x100, Scheme: SD2/monotonized_central
Starting 2D simulation: Euler 2D Riemann
Grid: 200x200, Scheme: SD2/monotonized_central
--------------------------------------------------
Анализ самосходимости (Экстраполяция Ричардсона)
Коэффициент измельчения (r) = 2.0
--------------------------------------------------

--- Density (rho) ---
Норма L1:
  Ошибка (Крупная-Средняя): 1.8603e-02
  Ошибка (Средняя-Мелкая):  1.1019e-02
  Наблюдаемый порядок (p):  0.7555
Норма L2:
  Ошибка (Крупная-Средняя): 4.1690e-02
  Ошибка (Средняя-Мелкая):  2.7567e-02
  Наблюдаемый порядок (p):  0.5967
Норма Linf:
  Ошибка (Крупная-Средняя): 3.0981e-01
  Ошибка (Средняя-Мелкая):  2.5495e-01
  Наблюдаемый порядок (p):  0.2812

--- Momentum X (rho*u) ---
Норма L1:
  Ошибка (Крупная-Средняя): 1.5974e-02
  Ошибка (Средняя-Мелкая):  1.0854e-02
  Наблюдаемый порядок (p):  0.5575
Норма L2

In [6]:
def run_with_different_mesh(grids: list[int], equation, limiter: str):
    """
    Запускает решатель последовательно на массиве различных сеток.

    :param grids: Список размеров сеток (например, [50, 100, 200])
    :param equation: Объект уравнения
    :param limiter: Название лимитера
    :return: Словарь с результатами и временем выполнения для каждой сетки
    """
    all_results = {}

    for J in grids:
        print(f"\n{'='*50}")
        print(f"Инициализация сетки: {J}x{J}")
        print(f"{'='*50}")

        # 1. Создаем параметры для текущей сетки
        pars = Pars2d(
            x_init=0.0, x_final=1.0,
            y_init=0.0, y_final=1.0,
            t_final=0.4, dt_out=0.005,
            Jx=J, Jy=J, cfl=0.475, scheme="sd2"
        )

        # 2. Инициализируем решатель
        solver = FastSolver2d(pars, equation, limiter_name=limiter)

        # 3. Прогрев JAX под новые размеры массивов (shape)
        print(f"--- Прогрев JAX (Компиляция XLA) для {J}x{J} ---")
        x_1d = jnp.linspace(pars.x_init + pars.dx / 2, pars.x_final - pars.dx / 2, pars.Jx)
        y_1d = jnp.linspace(pars.y_init + pars.dy / 2, pars.y_final - pars.dy / 2, pars.Jy)
        X, Y = jnp.meshgrid(x_1d, y_1d, indexing='ij')

        test_u = equation.initial_data(X, Y)

        # Прогреваем только один шаг, чтобы JAX скомпилировал XLA-граф
        _ = solver.update_step_jit(0.0, test_u, 0.001).block_until_ready()
        print("Прогрев завершен.\n")

        # 4. Основной запуск и бенчмарк
        print(f"--- Запуск JAX GPU Бенчмарка ---")
        t0 = time.time()

        results = solver.solve()
        results['u'][-1].block_until_ready() # Ждем завершения вычислений на GPU

        t1 = time.time()
        elapsed_time = t1 - t0

        print(f"[GPU JAX] Чистое время выполнения для {J}x{J}: {elapsed_time:.4f} секунд")

        # 5. Сохраняем результаты
        all_results[J] = {
            'time': elapsed_time,
            'u_final': results['u'][-1],
            'results_full': results
        }

    return all_results

if __name__ == "__main__":
    # Включаем двойную точность (обязательно для численных методов и экстраполяции)
    jax.config.update("jax_enable_x64", True)

    # Задаем массив сеток, по которым хотим пройтись
    grid_sizes = [40, 80, 160, 200, 320]

    # Создаем уравнение
    eqn = make_euler_riemann_2d()

    # Запускаем наш цикл
    run_results = run_with_different_mesh(grid_sizes, eqn, "minmod")

    # Если дальше вам нужно использовать эти данные для Ричардсона,
    # вы можете достать их из словаря:
    # u_c = run_results[50]['u_final']
    # u_m = run_results[100]['u_final']
    # u_f = run_results[200]['u_final']


Инициализация сетки: 40x40
--- Прогрев JAX (Компиляция XLA) для 40x40 ---
Прогрев завершен.

--- Запуск JAX GPU Бенчмарка ---
Starting 2D simulation: Euler 2D Riemann
Grid: 40x40, Scheme: SD2/minmod
[GPU JAX] Чистое время выполнения для 40x40: 0.2829 секунд

Инициализация сетки: 80x80
--- Прогрев JAX (Компиляция XLA) для 80x80 ---
Прогрев завершен.

--- Запуск JAX GPU Бенчмарка ---
Starting 2D simulation: Euler 2D Riemann
Grid: 80x80, Scheme: SD2/minmod
[GPU JAX] Чистое время выполнения для 80x80: 0.4103 секунд

Инициализация сетки: 160x160
--- Прогрев JAX (Компиляция XLA) для 160x160 ---
Прогрев завершен.

--- Запуск JAX GPU Бенчмарка ---
Starting 2D simulation: Euler 2D Riemann
Grid: 160x160, Scheme: SD2/minmod
[GPU JAX] Чистое время выполнения для 160x160: 0.8466 секунд

Инициализация сетки: 200x200
--- Прогрев JAX (Компиляция XLA) для 200x200 ---
Прогрев завершен.

--- Запуск JAX GPU Бенчмарка ---
Starting 2D simulation: Euler 2D Riemann
Grid: 200x200, Scheme: SD2/minmod
[GPU JAX]